# 35. Multi-Token Speculative Decoding | 多 Token 投机解码
**难度：** Hard | **环境：** CPU-first | **标签：** `推理优化`, `投机解码`, `Multi-Token Decoding` | **目标人群：** 推理优化学习者

---

## 本节导读

经典投机解码由独立草稿模型提出候选；另一类策略会通过多 token 预测头、多个候选或其他并行路径，在一轮中尝试推进更多 token。它们同样需要验证与回退，但候选来源、验证组织和质量保证方式可能不同。

本节把多 Token 推进作为投机式生成的一条策略分支：先观察候选如何形成，再看连续接受、首次拒绝和回退如何决定有效推进量，最后把候选长度与验证成本放进同一条收益判断。

**关键词：** `multi-token decoding`, `speculative decoding`, `verification`, `rollback`

## 前置阅读

**导语：** 先理解单 token 解码怎样更新状态，以及投机验证怎样按位置推进；再观察一次请求如何在一轮中处理多个候选 token。
- [21. Decoding Strategies | 解码策略](./21_Decoding_Strategies.ipynb)
- [23. Speculative Decoding | 投机解码](./23_Speculative_Decoding.ipynb)

---


### Step 1: 多 Token 推进在投机策略谱系中的位置

多 Token 解码与经典投机解码共享“提出候选 → 目标模型验证 → 接受或回退”的主线，区别主要在候选如何产生。放回 Task2 看，21 提供 Decode 阶段基础，23 解释经典投机语义，35 关注多 Token 候选的验证与回退，68 再用统一 workload 判断实际收益。

一次验证先形成候选序列，再从左到右确认可接受的连续前缀。前缀越长，单轮有效推进越多；较早发生拒绝时，后续候选不能直接沿用，需要回退并进入下一轮。DFlash、DSpark 等方法属于候选生成侧的扩展，仍然要经过同一验证与状态更新闭环。

图中上方区分候选产生方式，下方展开本节共享的执行闭环：候选形成后进入目标模型验证，再根据接受前缀或首次拒绝更新解码状态；状态随后反馈到下一轮候选形成。

| 路径 | 候选如何产生 | 验证后的状态 | 本节关注点 |
|---|---|---|---|
| 经典投机解码 | 独立草稿模型提出候选 | 概率接受、residual correction 或 bonus | 经典投机语义，详见 [23](./23_Speculative_Decoding.ipynb) |
| Multi-Token 推进 | 多 token 预测头或并行候选序列 | 连续接受前缀、首次拒绝和回退 | 本节的核心机制 |
| DFlash | block 级方式形成候选 | 候选长度与验证代价 | block 级候选策略 |
| DSpark | 半自回归方式结合置信度形成候选 | 候选长度、置信度与验证代价 | 置信度控制策略 |
| 项目验证 | 在统一 workload 下比较候选、验证与服务结果 | acceptance、质量、成本和端到端收益 | 由 [68](./68_Speculative_Decoding_Benchmark.ipynb) 收口 |

![投机解码与 Multi-Token 解码的关系](../docs/public/02_PyTorch_Algorithms/35_multi_token_overview.svg)

### Step 2: 候选序列与验证状态

先看一轮验证中有哪些信息，再理解状态如何变化。候选序列说明本轮准备尝试哪些 token；记目标模型对候选 token 给出的概率分布为 p，草稿路径给出的概率分布为 q，二者用于比较候选是否可靠。验证从左到右进行，首次拒绝会把序列分成可接受前缀和待回退后缀。


| 机制阶段 | 关键对象 | 作用 |
|------|----------|------|
| 提议序列 | 候选 token 与本轮候选上限 | 确定本轮最多尝试的范围 |
| 接受判断 | 草稿概率与目标概率 | 判断单个候选 token 是否可靠 |
| 顺序验证 | 候选序列与当前前缀 | 从左到右验证，首次拒绝后停止 |
| 结果汇总 | 接受前缀、拒绝位置、回退后缀 | 区分可直接推进与需要重新生成的部分 |


### Step 3: 连续接受、首次拒绝与有效推进

多 Token 解码的收益取决于连续接受长度，而不是候选数量本身。若一轮提出四个候选、前面三个通过验证，本轮有效推进量就是三个；若第一个就被拒绝，候选长度再长也不能带来有效推进。接受比例可以帮助比较候选质量，但最终还要结合目标模型验证和回退成本。

因此，DFlash 和 DSpark 的比较不能只看接受率：还要记录候选形成方式、候选长度、置信度预算、验证耗时和后续重生成代价。只有有效推进量能够抵消额外验证成本时，多 Token 路径才有端到端收益。

| 机制因素 | 偏大时的风险 | 偏小时的限制 |
|------|--------------|--------------|
| 候选长度 | 尝试范围更大，但拒绝和回退可能增加 | 单轮推进短，潜在收益有限 |
| 接受阈值 | 判断更严格，连续接受长度可能下降 | 判断更宽松，错误候选或重生成代价可能增加 |

![多 Token 解码的接受长度与收益](../docs/public/02_PyTorch_Algorithms/35_acceptance_budget.svg)


### Step 4: 实现多 Token 候选验证

下面的 `MultiTokenDecoderSim` 用一轮候选验证演示三个机制：候选长度怎样限制本轮工作量、单个位置怎样给出接受结果、首次拒绝怎样截断可推进前缀。骨架负责汇总接受前缀、拒绝位置和回退后缀，让你把注意力放在状态变化上。

| 实现部分 | TODO 关注点 | 结果检查 |
|---|---|---|
| TODO 1：候选前缀 | 按 `max_proposal_len` 构造本轮候选 | 不修改输入，长度不超限 |
| TODO 2：位置接受 | 依据草稿 / 目标概率和阈值判断当前位置 | 低草稿概率分支可解释 |
| TODO 3：连续前缀 | 接受时继续；首次拒绝时记录位置并停止 | 后续候选不能进入接受前缀 |
| 已给出汇总骨架 | 生成回退后缀和本轮推进比例 | 全部接受与拒绝两种结果一致 |


In [ ]:
from typing import List, Sequence, Tuple
import torch

In [ ]:
class MultiTokenDecoderSim:
    """教学用的多 token 提议—验证模拟器。

    用于观察候选长度、接受前缀和回退后缀如何共同决定一轮推进结果。"""

    def __init__(self, max_proposal_len: int = 4, min_accept_ratio: float = 0.5):
        if not isinstance(max_proposal_len, int) or isinstance(max_proposal_len, bool) or max_proposal_len <= 0:
            raise ValueError("max_proposal_len must be positive")
        if not (0.0 < min_accept_ratio <= 1.0):
            raise ValueError("min_accept_ratio must be in (0, 1]")
        self.max_proposal_len = max_proposal_len
        self.min_accept_ratio = min_accept_ratio
        self.history: List[dict] = []

    def propose(self, draft_tokens: Sequence[int]) -> List[int]:
        """把草稿序列截断为本轮允许验证的候选前缀。

        空序列返回空列表；返回长度不超过 `max_proposal_len`。"""
        # ==========================================
        # TODO 1: 从草稿 token 中生成本轮候选序列
        # 提示：先把 draft_tokens 转成 list，再截取前 max_proposal_len 个 token。
        # 这个结果决定本轮最多验证多少个位置，不要修改输入序列。
        # ==========================================
        # proposed = ???
        return proposed

    def _accept_token(self, draft_prob: float, target_prob: float) -> bool:
        """按教学用概率阈值判断一个候选是否通过验证。

        这里的阈值规则用于观察控制流，不等同于严格的分布保持算法。"""
        # ==========================================
        # TODO 2: 判断单个候选 token 是否被目标模型接受
        # 提示：正常情况下，target_prob 至少要达到
        # draft_prob * min_accept_ratio；draft_prob <= 0 时单独处理，并保持布尔返回值。
        # ==========================================
        if draft_prob <= 0:
            return target_prob > 0
        # accepted = ???
        return accepted

    def verify(
        self,
        draft_probs: torch.Tensor,
        target_probs: torch.Tensor,
        draft_tokens: Sequence[int],
    ) -> Tuple[List[int], int | None]:
        """从左到右验证候选，并在首次拒绝处停止。

        `draft_probs` 和 `target_probs` 的第 i 行对应第 i 个候选位置；
        返回已接受前缀以及首次拒绝的位置。"""
        proposed = self.propose(draft_tokens)
        accepted_tokens: List[int] = []
        rejected_at = None
        draft_probs = torch.as_tensor(draft_probs)
        target_probs = torch.as_tensor(target_probs)

        if draft_probs.ndim != 2 or target_probs.ndim != 2:
            raise ValueError("draft_probs 和 target_probs 必须是二维张量")
        if draft_probs.shape != target_probs.shape or draft_probs.shape[0] < len(proposed):
            raise ValueError("概率矩阵必须具有相同形状，并覆盖所有候选位置")
        vocab_size = draft_probs.shape[1]
        if any(not isinstance(token_id, int) or not 0 <= token_id < vocab_size for token_id in proposed):
            raise ValueError("draft token id 必须是词表范围内的整数")

        for i, token_id in enumerate(proposed):
            draft_prob = float(draft_probs[i, token_id])
            target_prob = float(target_probs[i, token_id])
            # ==========================================
            # TODO 3: 维护连续接受前缀
            # 提示：先得到 accepted。若为 True，将 token 加入 accepted_tokens；
            #       若为 False，记录当前 i 为 rejected_at 并停止，后续候选不能继续加入。
            # ==========================================
            # accepted = ???

            if accepted:
                accepted_tokens = accepted_tokens + [token_id]
            else:
                rejected_at = i
                break

        return accepted_tokens, rejected_at

    def decode(
        self,
        draft_probs: torch.Tensor,
        target_probs: torch.Tensor,
        draft_tokens: Sequence[int],
    ) -> dict:
        """汇总接受前缀、拒绝位置、回退后缀和本轮推进比例。

        `progress_per_round` 只表示本轮接受比例，不代表真实吞吐提升。"""
        proposed = self.propose(draft_tokens)
        accepted_tokens, rejected_at = self.verify(draft_probs, target_probs, draft_tokens)
        rejected_suffix = proposed[rejected_at:] if rejected_at is not None else []

        result = {
            "proposed_tokens": proposed,
            "accepted_tokens": accepted_tokens,
            "accepted_len": len(accepted_tokens),
            # 本轮真正推进的 token 占提议长度的比例，和项目 benchmark 的指标对应
            "progress_per_round": (len(accepted_tokens) / len(proposed)) if proposed else 0.0,
            "rejected_at": rejected_at,
            "rejected_suffix": rejected_suffix,
        }
        self.history.append(result)
        return result


In [ ]:
# 按机制拆分测试，分别定位候选提议、接受判断、首次拒绝和回退状态。
def _build_multi_token_fixture():
    draft_tokens = [10, 20, 30, 31]
    draft_probs = torch.zeros(4, 40)
    target_probs = torch.zeros(4, 40)
    for index, token in enumerate(draft_tokens):
        draft_probs[index, token] = 0.5
        target_probs[index, token] = 0.8 if index < 2 else 0.2
    return draft_tokens, draft_probs, target_probs

def test_proposal_contract():
    draft_tokens, _, _ = _build_multi_token_fixture()
    sim = MultiTokenDecoderSim(max_proposal_len=3, min_accept_ratio=0.6)
    assert sim.propose(draft_tokens) == [10, 20, 30]

def test_acceptance_rule():
    sim = MultiTokenDecoderSim(max_proposal_len=3, min_accept_ratio=0.6)
    assert sim._accept_token(0.5, 0.31) is True
    assert sim._accept_token(0.5, 0.2) is False

def test_first_rejection():
    draft_tokens, draft_probs, target_probs = _build_multi_token_fixture()
    sim = MultiTokenDecoderSim(max_proposal_len=3, min_accept_ratio=0.6)
    accepted, rejected_at = sim.verify(draft_probs, target_probs, draft_tokens)
    assert accepted == [10, 20]
    assert rejected_at == 2

def test_rejection_closes_the_suffix():
    draft_tokens, draft_probs, target_probs = _build_multi_token_fixture()
    target_probs[3, draft_tokens[3]] = 0.9  # 后一个候选即使满足阈值，也不能越过第 3 个位置的拒绝。
    sim = MultiTokenDecoderSim(max_proposal_len=4, min_accept_ratio=0.6)
    accepted, rejected_at = sim.verify(draft_probs, target_probs, draft_tokens)
    assert accepted == [10, 20] and rejected_at == 2

def test_rollback_suffix():
    draft_tokens, draft_probs, target_probs = _build_multi_token_fixture()
    sim = MultiTokenDecoderSim(max_proposal_len=3, min_accept_ratio=0.6)
    result = sim.decode(draft_probs, target_probs, draft_tokens)
    assert result["proposed_tokens"] == [10, 20, 30]
    assert result["accepted_tokens"] == [10, 20]
    assert result["accepted_len"] == 2
    assert result["progress_per_round"] == 2 / 3
    assert result["rejected_at"] == 2
    assert result["rejected_suffix"] == [30]
    assert len(sim.history) == 1

def test_all_accept_and_variable_length():
    draft_tokens, draft_probs, _ = _build_multi_token_fixture()
    all_target = torch.zeros(4, 40)
    for index, token in enumerate(draft_tokens):
        all_target[index, token] = 0.8
    for proposal_len in (1, 2, 3, 4):
        sim = MultiTokenDecoderSim(max_proposal_len=proposal_len, min_accept_ratio=0.6)
        result = sim.decode(draft_probs, all_target, draft_tokens)
        assert result["accepted_len"] == proposal_len
        assert result["progress_per_round"] == 1.0
        assert result["rejected_at"] is None
        assert result["rejected_suffix"] == []

def test_multi_token_decoder():
    try:
        test_proposal_contract()
        test_acceptance_rule()
        test_first_rejection()
        test_rejection_closes_the_suffix()
        test_rollback_suffix()
        test_all_accept_and_variable_length()
        print("✅ MultiTokenDecoderSim 机制测试通过：提议、接受、拒绝和回退均符合预期。")
    except NotImplementedError:
        print("请先完成 TODO 部分的代码！")
        raise
    except Exception as error:
        print(f"❌ MultiTokenDecoderSim 测试失败: {type(error).__name__}: {error}")
        raise


test_multi_token_decoder()


## 参考代码与解析

### 代码


In [ ]:
class MultiTokenDecoderSim:
    """教学用的多 token 提议—验证模拟器。

    用于观察候选长度、接受前缀和回退后缀如何共同决定一轮推进结果。"""

    def __init__(self, max_proposal_len: int = 4, min_accept_ratio: float = 0.5):
        if not isinstance(max_proposal_len, int) or isinstance(max_proposal_len, bool) or max_proposal_len <= 0:
            raise ValueError("max_proposal_len must be positive")
        if not (0.0 < min_accept_ratio <= 1.0):
            raise ValueError("min_accept_ratio must be in (0, 1]")
        self.max_proposal_len = max_proposal_len
        self.min_accept_ratio = min_accept_ratio
        self.history: List[dict] = []

    def propose(self, draft_tokens: Sequence[int]) -> List[int]:
        """把草稿序列截断为本轮允许验证的候选前缀。

        空序列返回空列表；返回长度不超过 `max_proposal_len`。"""
        # ==========================================
        # TODO 1: 从草稿 token 中生成本轮候选序列
        # 提示：先把 draft_tokens 转成 list，再截取前 max_proposal_len 个 token。
        # 这个结果决定本轮最多验证多少个位置，不要修改输入序列。
        # ==========================================
        proposed = list(draft_tokens)[: self.max_proposal_len]
        return proposed

    def _accept_token(self, draft_prob: float, target_prob: float) -> bool:
        """按教学用概率阈值判断一个候选是否通过验证。

        这里的阈值规则用于观察控制流，不等同于严格的分布保持算法。"""
        # ==========================================
        # TODO 2: 判断单个候选 token 是否被目标模型接受
        # 提示：正常情况下，target_prob 至少要达到
        # draft_prob * min_accept_ratio；draft_prob <= 0 时单独处理，并保持布尔返回值。
        # ==========================================
        if draft_prob <= 0:
            return target_prob > 0
        accepted = target_prob >= draft_prob * self.min_accept_ratio
        return accepted

    def verify(
        self,
        draft_probs: torch.Tensor,
        target_probs: torch.Tensor,
        draft_tokens: Sequence[int],
    ) -> Tuple[List[int], int | None]:
        """从左到右验证候选，并在首次拒绝处停止。

        `draft_probs` 和 `target_probs` 的第 i 行对应第 i 个候选位置；
        返回已接受前缀以及首次拒绝的位置。"""
        proposed = self.propose(draft_tokens)
        accepted_tokens: List[int] = []
        rejected_at = None
        draft_probs = torch.as_tensor(draft_probs)
        target_probs = torch.as_tensor(target_probs)

        if draft_probs.ndim != 2 or target_probs.ndim != 2:
            raise ValueError("draft_probs 和 target_probs 必须是二维张量")
        if draft_probs.shape != target_probs.shape or draft_probs.shape[0] < len(proposed):
            raise ValueError("概率矩阵必须具有相同形状，并覆盖所有候选位置")
        vocab_size = draft_probs.shape[1]
        if any(not isinstance(token_id, int) or not 0 <= token_id < vocab_size for token_id in proposed):
            raise ValueError("draft token id 必须是词表范围内的整数")

        for i, token_id in enumerate(proposed):
            draft_prob = float(draft_probs[i, token_id])
            target_prob = float(target_probs[i, token_id])
            # ==========================================
            # TODO 3: 维护连续接受前缀
            # 提示：先得到 accepted。若为 True，将 token 加入 accepted_tokens；
            #       若为 False，记录当前 i 为 rejected_at 并停止，后续候选不能继续加入。
            # ==========================================
            accepted = self._accept_token(draft_prob, target_prob)

            if accepted:
                accepted_tokens = accepted_tokens + [token_id]
            else:
                rejected_at = i
                break

        return accepted_tokens, rejected_at

    def decode(
        self,
        draft_probs: torch.Tensor,
        target_probs: torch.Tensor,
        draft_tokens: Sequence[int],
    ) -> dict:
        """汇总接受前缀、拒绝位置、回退后缀和本轮推进比例。

        `progress_per_round` 只表示本轮接受比例，不代表真实吞吐提升。"""
        proposed = self.propose(draft_tokens)
        accepted_tokens, rejected_at = self.verify(draft_probs, target_probs, draft_tokens)
        rejected_suffix = proposed[rejected_at:] if rejected_at is not None else []

        result = {
            "proposed_tokens": proposed,
            "accepted_tokens": accepted_tokens,
            "accepted_len": len(accepted_tokens),
            # 本轮真正推进的 token 占提议长度的比例，和项目 benchmark 的指标对应
            "progress_per_round": (len(accepted_tokens) / len(proposed)) if proposed else 0.0,
            "rejected_at": rejected_at,
            "rejected_suffix": rejected_suffix,
        }
        self.history.append(result)
        return result


### 解析

**TODO 1：构造候选前缀**
- 将草稿 token 复制为列表后按 `max_proposal_len` 截取。候选长度只规定本轮最多验证多少个位置，不改变原始草稿序列。

**TODO 2：判断当前位置是否接受**
- 当 `draft_prob > 0` 时，比较 `target_prob` 与 `draft_prob * min_accept_ratio`；草稿概率为零时单独处理，避免把无意义的比例带入判断。

**TODO 3：维护连续接受前缀**
- 验证顺序决定状态：当前 token 被接受才可继续；首次拒绝的位置写入 `rejected_at` 后立即停止。这样 `accepted_tokens` 始终是候选序列的连续前缀，`rejected_suffix` 则由骨架据此生成。

| 观察量 | 说明 | 在后续基准中的作用 |
|---|---|---|
| `max_proposal_len` | 每轮最多尝试的候选数 | 影响可获得的推进空间 |
| `accepted_len` / `progress_per_round` | 本轮实际被连续接受的前缀 | 反映候选质量与回退压力 |
| `rejected_at` / `rejected_suffix` | 首次拒绝及其后的候选 | 说明下一轮需要重新处理的部分 |



### Step 5: GPU 可选实验——批量候选验证探针

使用固定的 draft/target logits，在 GPU 内比较逐位置前缀扫描与一次性批量前缀计算。两条路径应得到相同的连续接受长度；差异用于观察候选验证的局部调度与张量路径。

| 实验对象 | baseline | candidate | 证据范围 |
|---|---|---|---|
| 候选验证 | GPU 内逐位置前缀扫描 | GPU 内一次性批量前缀计算 | 单 GPU synthetic verification probe |


#### 5.1 配置与 workload

固定 batch、候选长度、词表大小、warmup、重复次数和随机种子。baseline 与 candidate 都只处理同一份 logits，避免把 host-device 同步混入对照。

In [ ]:
# GPU 探针配置：默认关闭；只测同一份 logits 上的候选验证路径。
RUN_GPU_EXPERIMENT = False
GPU_BATCH_SIZE = 32
GPU_PROPOSAL_LEN = 4
GPU_VOCAB_SIZE = 4096
GPU_WARMUP = 10
GPU_ITERS = 50
GPU_MIN_ACCEPT_RATIO = 0.6
GPU_RESULT_PATH = 'benchmarks/results/35_multi_token_gpu_smoke.json'

#### 5.2 执行与保存 JSON

开启后，代码先生成固定 logits，再计算两种实现的连续接受前缀长度并核对一致性，最后保存局部耗时与证据等级。


In [ ]:
import json
import time
from pathlib import Path

def _run_multi_token_gpu_smoke():
    if not RUN_GPU_EXPERIMENT:
        print('已跳过 GPU smoke：将 RUN_GPU_EXPERIMENT 改为 True 后重新运行。')
        return None
    if not torch.cuda.is_available():
        raise RuntimeError('需要 CUDA GPU 才能运行本实验。')
    if min(GPU_BATCH_SIZE, GPU_PROPOSAL_LEN, GPU_VOCAB_SIZE, GPU_WARMUP, GPU_ITERS) <= 0:
        raise ValueError('batch、proposal length、vocab、warmup 和 iters 必须为正数。')
    device = torch.device('cuda')
    generator = torch.Generator(device=device).manual_seed(35)
    draft_logits = torch.randn(GPU_BATCH_SIZE, GPU_PROPOSAL_LEN, GPU_VOCAB_SIZE, device=device, generator=generator)
    target_logits = torch.randn(GPU_BATCH_SIZE, GPU_PROPOSAL_LEN, GPU_VOCAB_SIZE, device=device, generator=generator)
    draft_tokens = draft_logits.argmax(dim=-1)
    draft_prob = draft_logits.softmax(dim=-1).gather(-1, draft_tokens.unsqueeze(-1)).squeeze(-1)
    target_prob = target_logits.softmax(dim=-1).gather(-1, draft_tokens.unsqueeze(-1)).squeeze(-1)
    accepted = target_prob >= draft_prob * GPU_MIN_ACCEPT_RATIO

    def tokenwise_prefix_scan():
        """逐位置更新仍处于可接受前缀中的样本，不把 GPU 标量读回 CPU。"""
        active = torch.ones(GPU_BATCH_SIZE, dtype=torch.bool, device=device)
        prefix_len = torch.zeros(GPU_BATCH_SIZE, dtype=torch.int64, device=device)
        for index in range(GPU_PROPOSAL_LEN):
            accepted_here = active & accepted[:, index]
            prefix_len = prefix_len + accepted_here.to(prefix_len.dtype)
            active = accepted_here
        return prefix_len

    def batched_prefix_scan():
        """一次性定位每个样本的首次拒绝位置，得到连续接受前缀长度。"""
        rejected = ~accepted
        first_rejected = rejected.to(torch.int64).argmax(dim=1)
        all_accepted = ~rejected.any(dim=1)
        return torch.where(all_accepted, torch.full_like(first_rejected, GPU_PROPOSAL_LEN), first_rejected)

    def measure(fn):
        for _ in range(GPU_WARMUP):
            fn()
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(GPU_ITERS):
            fn()
        torch.cuda.synchronize()
        return (time.perf_counter() - start) * 1000 / GPU_ITERS

    baseline_prefix_len = tokenwise_prefix_scan()
    candidate_prefix_len = batched_prefix_scan()
    if not torch.equal(baseline_prefix_len, candidate_prefix_len):
        raise RuntimeError('两条验证路径的连续接受前缀长度不一致。')
    acceptance_rate = float(accepted.float().mean().item())
    mean_prefix_len = float(candidate_prefix_len.float().mean().item())
    baseline_ms = measure(tokenwise_prefix_scan)
    candidate_ms = measure(batched_prefix_scan)
    result = {
        'schema_version': 'multi-token-gpu-smoke/v1',
        'experiment': '35_multi_token_verification',
        'project': '35',
        'role': 'candidate_verification_path',
        'status': 'completed',
        'workload': {'batch_size': GPU_BATCH_SIZE, 'proposal_length': GPU_PROPOSAL_LEN, 'vocab_size': GPU_VOCAB_SIZE, 'warmup': GPU_WARMUP, 'iters': GPU_ITERS},
        'config': {'min_accept_ratio': GPU_MIN_ACCEPT_RATIO, 'seed': 35, 'device_type': 'cuda'},
        'baseline': {'name': 'tokenwise_gpu_prefix_scan', 'verify_ms': round(baseline_ms, 6)},
        'candidate': {'name': 'batched_gpu_prefix_scan', 'verify_ms': round(candidate_ms, 6)},
        'metrics': {'acceptance_rate': acceptance_rate, 'mean_accepted_prefix_len': mean_prefix_len, 'speedup': round(baseline_ms / candidate_ms, 6) if candidate_ms else None},
        'strategy_metrics': {'baseline_verify_ms': baseline_ms, 'candidate_verify_ms': candidate_ms},
        'environment': {'device': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'cuda': torch.version.cuda},
        'artifact': {'result_path': GPU_RESULT_PATH},
        'quality': {'status': 'not_evaluated', 'output_consistency': 'not_applicable_for_synthetic_logits'},
        'failure': None,
        'evidence_level': 'single_gpu_synthetic_verification_probe',
        'decision': {'decision': 'measure', 'reason': '局部验证路径可继续比较；真实 draft/target 模型的端到端收益交由 68 验证'},
    }
    path = Path(GPU_RESULT_PATH)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result

multi_token_gpu_result = _run_multi_token_gpu_smoke()

#### 5.3 读取结果与对照

先核对两条路径的 `mean_accepted_prefix_len` 是否来自同一份 logits，再比较 `baseline.verify_ms`、`candidate.verify_ms` 和 `speedup`。`quality` 保持未评估；真实 draft/target 模型的输出一致性与服务吞吐由 68 的统一 workload 记录。


In [ ]:
# 5.3：读取 GPU smoke JSON；默认未运行时保留待复测状态。
import json
from pathlib import Path
result_path = Path(GPU_RESULT_PATH)
if result_path.exists():
    result = json.loads(result_path.read_text(encoding='utf-8'))
    print(result)
else:
    print(f'尚无 GPU smoke 结果：{result_path}')


#### 5.4 结果解释与下一步

将探针结果与 68 的真实模型 benchmark 分开解读：本节的对照只改变验证张量的组织方式，68 才同时测量草稿、目标验证、KV Cache 与服务调度。

| 观察字段 | 可以回答的问题 | 后续动作 |
|---|---|---|
| `mean_accepted_prefix_len`、`acceptance_rate` | 固定 logits 下候选平均能推进多长 | 仅作路径一致性与工作量记录 |
| `baseline.verify_ms`、`candidate.verify_ms`、`speedup` | 批量前缀计算是否减少局部验证开销 | 若有收益，再进入 68 复测 |
| `quality`、`evidence_level`、`decision` | 结论覆盖的是哪一层证据 | 不把本表解释为模型质量或 serving 吞吐 |


## 相关阅读

完成候选生成、顺序验证和回退处理后，可以继续比较不同投机式生成路径的候选来源、验证成本与真实 benchmark。

- [Medusa 原论文：Simple LLM Inference Acceleration Framework with Multiple Decoding Heads](https://arxiv.org/abs/2401.10782)
- [Speculative Sampling 原论文](https://arxiv.org/abs/2302.01318)
- [vLLM Speculative Decoding 文档](https://docs.vllm.ai/en/latest/features/spec_decode.html)
- [扩展阅读：DFlash 论文：Block Diffusion for Flash Speculative Decoding](https://arxiv.org/abs/2602.06036)
- [扩展阅读：DSpark 论文：Confidence-Scheduled Speculative Decoding](https://arxiv.org/abs/2607.05147)
- [36. Decode Scheduling | Decode 调度](./36_Decode_Scheduling.ipynb)
- [68. Speculative Decoding Benchmark | 投机解码基准](./68_Speculative_Decoding_Benchmark.ipynb)